# 07 · `error_codes.json` — 에러코드 정의 (기능 1의 뿌리)

| | |
|---|---|
| **목적** | `lookup_error_code` 가 exact match 로 읽는 **결정론적 소스**(D1)의 무결성 점검 |
| **input** | `data/extracted/error_codes.json` (65건 · `extract_error_codes.py` 산출) |
| **output** | `data-analysis/data/processed/error_codes.parquet`<br>`data-analysis/reports/error_codes_decisions.json` |

> ⛔ **이 파일은 이미 사람 승인을 받은 산출물이다** (`_status: 승인 완료 2026-07-28`).
> 이 노트북은 **읽고 검사만 한다** — 값을 고치지 않는다. 정정은 `extract_error_codes.py` 재실행으로만.

> 📌 **왜 이제야 EDA 하는가** — 이 파일은 검증을 마치고 코드가 소비 중이라 그동안 EDA 대상이
> 아니었다. 다만 `99` 보고서의 **기능 1·5 칸이 비어 있어** 근거를 채우려고 뒤늦게 다룬다.

In [ ]:
import sys
sys.path.insert(0, "..")
import json

import pandas as pd
from common import EXTRACTED, profile, save_decisions, save_processed, top_values

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 1. 개요

| 항목 | 값 |
|---|---|
| 원천 | `iG5A_User_Manual_Standard_KR_210303.pdf` · `S100_Manual_Korean_V4.2.pdf` |
| 만든 것 | `data/extracted/extract_error_codes.py` |
| 승인 | **2026-07-28 사람 승인 완료** (`data/analysis/ig5a_code_mapping.md`) |
| 인용 기준 | **PDF 물리 페이지** (D26) — 표시 변환만 오프셋 적용(D32) |
| 코드 정책 | `code` = 대문자 canonical · `display_code` = 키패드 원표기 (**D25**) |

In [ ]:
raw = json.loads((EXTRACTED / "error_codes.json").read_text(encoding="utf-8"))
for k, v in raw.items():
    if k != "entries":
        print(f"{k:18}: {v}")
print(f"{'entries':18}: {len(raw['entries'])}건")

### 1-1. 🚨 선언된 건수와 실제 건수가 맞는가

In [ ]:
df_raw = pd.DataFrame(raw["entries"])
실제 = df_raw["model"].value_counts().to_dict()
선언 = raw["counts"]
print("선언 (counts 필드):", 선언)
print("실제 (entries 집계):", 실제)
print()
for m in 선언:
    ok = 선언[m] == 실제.get(m)
    print(f"  {m:6} 선언 {선언[m]:3} vs 실제 {실제.get(m, 0):3}  {'✅' if ok else '❌ 불일치'}")
assert 선언 == 실제, "counts 필드와 entries 가 어긋난다"
print(f"\n합계 {sum(실제.values())}건 — CLAUDE.md 가 회귀 기준으로 삼는 65 와 일치")

## 2. 로드 + 기초 통계

In [ ]:
print(f"원본 shape: {df_raw.shape[0]}행 × {df_raw.shape[1]}열")
print("컬럼:", list(df_raw.columns))

In [ ]:
profile(df_raw)

In [ ]:
df_raw.head(4)[["model", "code", "display_code", "error_name", "severity", "manual_page"]]

## 3. EDA

### 3-1. 🚨 `D6`·`D13` 의 실증 — 같은 코드가 두 기종에 모두 있다

`model` 파라미터를 `enum('iG5A','S100')` 으로 **강제**하는 이유가 여기 있다.
코드만으로 조회하면 어느 기종 것인지 알 수 없다.

In [ ]:
겹침 = df_raw.groupby("code")["model"].nunique()
공통 = sorted(겹침[겹침 > 1].index)
print(f"두 기종에 모두 있는 code: {len(공통)}종 / 전체 {df_raw['code'].nunique()}종")
print(공통)
print()
print(f"→ 코드만으로 조회하면 {len(공통)}종이 **어느 기종 것인지 결정되지 않는다.**")

### 3-2. ⚠️ 다만 정확히 말하면 — "다른 고장"이 아니라 "다른 표기·인용·심각도"다

문서가 종종 *"같은 코드, 다른 의미"* 로 요약하는데, 실측은 조금 다르다.
물리적 의미는 대체로 같고(과전류 = Over Current), **진짜로 갈리는 건 3가지**다.

In [ ]:
pair = df_raw[df_raw["code"].isin(공통)].pivot(
    index="code", columns="model",
    values=["display_code", "error_name", "manual_page", "severity"])
비교 = pd.DataFrame({
    "iG5A 표기": pair[("display_code", "iG5A")],
    "S100 표기": pair[("display_code", "S100")],
    "iG5A 페이지": pair[("manual_page", "iG5A")],
    "S100 페이지": pair[("manual_page", "S100")],
    "iG5A severity": pair[("severity", "iG5A")],
    "S100 severity": pair[("severity", "S100")],
})
비교["표기 다름"] = 비교["iG5A 표기"] != 비교["S100 표기"]
비교["severity 다름"] = 비교["iG5A severity"] != 비교["S100 severity"]
비교

In [ ]:
print(f"① display_code 표기가 다른 코드 : {비교['표기 다름'].sum()} / {len(비교)}건")
print("   → D25 의 canonical/display 분리가 없으면 키패드 표기로는 조회가 안 된다")
print()
print(f"② manual_page 가 다른 코드      : {(비교['iG5A 페이지'] != 비교['S100 페이지']).sum()} / {len(비교)}건")
print("   → iG5A 202~204 vs S100 416~419. **인용은 반드시 기종별로 갈라야 한다**")
print()
diff = 비교[비교["severity 다름"]]
print(f"③ severity 가 다른 코드         : {len(diff)}건")
if len(diff):
    print(diff[["iG5A severity", "S100 severity"]].to_string())
    print("   → 판정에 실제로 영향을 주는 유일한 차이다")

### 3-3. severity 분포

In [ ]:
print(top_values(df_raw, "severity").to_string(index=False))
print()
print(pd.crosstab(df_raw["model"], df_raw["severity"], margins=True).to_string())
print()
print("→ warning 10건은 전부 S100 이다. iG5A 매뉴얼에는 경보 표가 따로 없다.")

### 3-4. 🚨 `actions` 가 비어 있는 코드가 절반이 넘는다

S4(미지 코드) 흐름과 별개로, **조치 문구가 없는 코드**는 응답 품질에 직접 영향을 준다.

In [ ]:
df_raw["actions_수"] = df_raw["actions"].map(len)
df_raw["causes_수"] = df_raw["causes"].map(len)
빈 = df_raw["actions_수"] == 0
print(f"actions 빈 건: {빈.sum()} / {len(df_raw)}  ({빈.mean() * 100:.1f}%)")
print()
print(pd.crosstab(df_raw["model"], 빈.map({True: "actions 없음", False: "actions 있음"})).to_string())
print()
print("actions 없는 코드 (상위 12) —")
print(df_raw[빈][["model", "code", "error_name"]].head(12).to_string(index=False))

> ⚠ **`actions` 가 비었다고 데이터 결함은 아니다.** 매뉴얼 원문 자체가 조치란 없이
> 현상 설명만 있는 코드가 있다(예: `COL` 입력결상, `EEP` 파라미터 저장 이상).
> **없는 조치를 지어내면 안 된다**(D65 와 같은 태도) — 다만 응답 템플릿이 이 경우를
> 어떻게 다루는지는 별도 확인이 필요하다.

### 3-5. 🚨 스키마 비대칭 — `description` 은 S100 에만 있다

In [ ]:
있음 = df_raw["description"].notna()
print(pd.crosstab(df_raw["model"], 있음.map({True: "description 있음", False: "없음"})).to_string())
print()
print(f"→ description 은 S100 {있음.sum()}건에만 있다. iG5A 는 전건 없음.")
print("  같은 테이블인데 기종에 따라 필드가 있다 없다 한다 — 소비처가 None 을 다뤄야 한다.")

### 3-6. 중복 `error_name` — 같은 이름이 여러 코드에 붙는다

In [ ]:
dup = df_raw.groupby(["model", "error_name"])["code"].agg(list)
dup = dup[dup.map(len) > 1]
print("같은 error_name 을 공유하는 코드 —")
for (m, name), codes in dup.items():
    print(f"  {m:6} {name!r:24} → {codes}")
print()
print("→ S100 의 IO Board Trip 3종은 **표시 순환 상태**(Errc→-rrc→…)를 각각 코드로 잡은 것이다.")
print("  error_name 으로 조회하면 3건이 나오므로 **키는 (model, code) 여야 한다**(D13).")

### 3-7. `manual_page` 분포 — 인용 앵커

In [ ]:
print(df_raw.groupby("model")["manual_page"].agg(["min", "max", "nunique", "count"]).to_string())
print()
print("페이지별 코드 수 —")
print(df_raw.groupby(["model", "manual_page"]).size().to_string())

## 4. 전처리

⛔ **값을 고치지 않는다.** 리스트 컬럼(`causes`·`actions`)만 parquet 저장 가능한 형태로 바꾸고,
분석용 파생을 추가한다.

In [ ]:
df = df_raw.copy()

# 리스트 → 문자열 (parquet 저장 + 사람이 읽기 위해). 원본 개수는 별도 컬럼에 남긴다
df["causes_텍스트"] = df["causes"].map(lambda xs: "\n".join(xs))
df["actions_텍스트"] = df["actions"].map(lambda xs: "\n".join(xs))
df = df.drop(columns=["causes", "actions"])

df["조치있음"] = df["actions_수"] > 0
df["양기종공통"] = df["code"].isin(공통)
df["설명있음"] = df["description"].notna()
df["본문길이"] = df["causes_텍스트"].str.len() + df["actions_텍스트"].str.len()

print(f"정제 후: {df.shape[0]}행 × {df.shape[1]}열")
print(df[["조치있음", "양기종공통", "설명있음"]].sum().to_string())

## 5. 결정 로그

In [ ]:
column_decisions = [
    {"column": "model", "used": True,
     "reason": "🚨 복합키의 절반. enum 2종 강제(D6·D13) — 12종 코드가 두 기종에 겹쳐 이게 없으면 조회 불가"},
    {"column": "code", "used": True, "reason": "복합키의 나머지 절반. 대문자 canonical (D25)"},
    {"column": "display_code", "used": True,
     "reason": "키패드 원표기(D25). 사용자가 보는 건 'OCt' 지 'OCT' 가 아니다 — 12종 전부 표기가 다르다"},
    {"column": "error_name", "used": True,
     "reason": "응답 표시용. ⛔ 조회 키로는 못 쓴다 — S100 IO Board Trip 이 3개 코드에 중복"},
    {"column": "severity", "used": True, "reason": "fault/warning/critical. HWT 만 기종 간 값이 다르다"},
    {"column": "causes", "used": True, "reason": "원인 목록. 응답 본문"},
    {"column": "actions", "used": True, "reason": "조치 목록. 37건(56.9%)이 빈 배열 — 매뉴얼 원문이 그렇다"},
    {"column": "manual_page", "used": True,
     "reason": "🚨 인용 앵커. PDF 물리 페이지(D26). 기종별로 범위가 완전히 다르다(202~204 vs 416~419)"},
    {"column": "description", "used": True, "reason": "S100 41건에만 있다. 스키마 비대칭 — 소비처가 None 처리 필요"},
    {"column": "source", "used": False, "reason": "추출 출처 메모. 전건 기종별 상수라 변별력 0 — 계보는 _status 가 갖는다"},
    {"column": "mapping_confidence", "used": False,
     "reason": "iG5A 24건 전부 'high' 이고 S100 엔 필드 자체가 없다. 승인 완료 후라 판정에 쓰이지 않는다"},
]
assert len(column_decisions) == df_raw.shape[1] - 2, \
    f"결정 {len(column_decisions)} vs 원본 컬럼 {df_raw.shape[1] - 2} (파생 2 제외)"
print(f"컬럼 결정 {len(column_decisions)}건 — 원본 컬럼 수와 일치")

In [ ]:
save_decisions(
    "error_codes",
    column_decisions,
    source="LS 매뉴얼 PDF 2종 (iG5A 표준본 · S100 V4.2)",
    generator="data/extract_error_codes.py",
    approved_at="2026-07-28",
    row_count_raw=int(df_raw.shape[0]),
    col_count_raw=int(df_raw.shape[1] - 2),
    counts_declared=raw["counts"],
    counts_actual={k: int(v) for k, v in 실제.items()},
    shared_codes=공통,
    generated_columns=["causes_텍스트", "actions_텍스트", "causes_수", "actions_수",
                       "조치있음", "양기종공통", "설명있음", "본문길이"],
    caveats=[
        "⛔ 승인 완료 산출물이다 — 값을 고치지 않는다. 정정은 extract_error_codes.py 재실행으로만",
        "12종 코드가 두 기종에 모두 있다 — (model, code) 복합키가 아니면 조회가 결정되지 않는다(D13)",
        "다만 '같은 코드 다른 고장'은 과장이다. 물리적 의미는 대체로 같고 실제로 갈리는 것은 "
        "① display_code 표기(12/12) ② manual_page(12/12) ③ severity(HWT 1건뿐)",
        "actions 가 빈 코드가 37건(56.9%) — 매뉴얼 원문이 그렇다. 없는 조치를 지어내면 안 된다",
        "description 은 S100 41건에만 있다 (스키마 비대칭)",
        "error_name 은 조회 키로 쓸 수 없다 — S100 IO Board Trip 이 3개 코드에 중복",
    ],
)

## 6. 정제 데이터 저장

In [ ]:
save_processed("error_codes", df)